# NB08 — Citation Management with Zotero and BibTeX

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 1.5 hours  

---

## Motivation

Reference management is infrastructure. Done poorly, it consumes hours before a deadline. Done well, it is invisible. This notebook covers Zotero (free, open-source, the standard in academic science) and BibTeX (the standard citation format for LaTeX, Pandoc, and most journal submission systems).

## 1. Zotero Workflow

1. **Install:** Zotero desktop (https://www.zotero.org/) + Zotero Connector browser extension
2. **Capture:** When you find a paper on PubMed, bioRxiv, or a journal site, click the Connector icon — Zotero pulls metadata automatically
3. **Organize:** Create Collections (folders) per module. A paper can belong to multiple collections.
4. **Annotate:** Add tags and notes directly in Zotero — these persist with the entry
5. **Export:** File → Export Library → BibTeX format (.bib file) for use with LaTeX/Pandoc
6. **Sync:** Zotero sync (free up to 300 MB) or WebDAV for PDF storage

**Key shortcut:** If you have a DOI, paste it into Zotero (Add item by identifier) and it fetches all metadata.

## 2. BibTeX Anatomy

```bibtex
@article{needleman1970,
  author  = {Needleman, Saul B. and Wunsch, Christian D.},
  title   = {A general method applicable to the search for similarities in the amino acid sequence of two proteins},
  journal = {Journal of Molecular Biology},
  year    = {1970},
  volume  = {48},
  number  = {3},
  pages   = {443--453},
  doi     = {10.1016/0022-2836(70)90057-4},
}
```

### Required vs optional fields by entry type

| Type | Required | Commonly needed |
|------|----------|------------------|
| `@article` | author, title, journal, year | volume, number, pages, doi |
| `@book` | author, title, publisher, year | edition, address, isbn |
| `@inproceedings` | author, title, booktitle, year | pages, publisher, doi |
| `@preprint` | author, title, year | doi, url, note ("bioRxiv preprint") |
| `@software` | author, title, year | url, version, note |
| `@phdthesis` | author, title, school, year | address, type |

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np


REQUIRED_FIELDS = {
    'article': {'author', 'title', 'journal', 'year'},
    'book': {'author', 'title', 'publisher', 'year'},
    'inproceedings': {'author', 'title', 'booktitle', 'year'},
    'preprint': {'author', 'title', 'year'},
    'software': {'author', 'title', 'year'},
    'phdthesis': {'author', 'title', 'school', 'year'},
    'misc': {'author', 'title', 'year'},
}


@dataclass
class BibEntry:
    entry_type: str
    cite_key: str
    fields: Dict[str, str]

    def missing_required(self) -> List[str]:
        required = REQUIRED_FIELDS.get(self.entry_type.lower(), {'author', 'title', 'year'})
        return [f for f in required if f not in self.fields]

    def has_doi(self) -> bool:
        return 'doi' in self.fields and bool(self.fields['doi'].strip())

    def format_apa(self) -> str:
        """Format entry as APA 7th edition citation string."""
        f = self.fields
        authors = f.get('author', 'Unknown')
        year = f.get('year', 'n.d.')
        title = f.get('title', 'Untitled')
        journal = f.get('journal', '')
        volume = f.get('volume', '')
        pages = f.get('pages', '')
        doi = f.get('doi', '')

        author_str = authors.split(' and ')[0].split(',')[0]
        if ' and ' in authors:
            n_authors = len(authors.split(' and '))
            if n_authors > 1:
                author_str += f' et al.'

        citation = f'{author_str} ({year}). {title}.'
        if journal:
            citation += f' {journal}'
            if volume:
                citation += f', {volume}'
            if pages:
                citation += f', {pages}'
            citation += '.'
        if doi:
            citation += f' https://doi.org/{doi}'
        return citation

    def format_nature(self) -> str:
        """Format entry as Nature-style citation (Author et al. Journal Vol, Pages (Year))."""
        f = self.fields
        authors = f.get('author', 'Unknown')
        first_author = authors.split(' and ')[0].split(',')[0]
        et_al = ' et al.' if ' and ' in authors else ''
        year = f.get('year', 'n.d.')
        title = f.get('title', 'Untitled')
        journal = f.get('journal', '')
        volume = f.get('volume', '')
        pages = f.get('pages', '')
        doi = f.get('doi', '')

        citation = f'{first_author}{et_al}. {title}. '
        if journal:
            citation += f'*{journal}*'
            if volume:
                citation += f' **{volume}**'
            if pages:
                citation += f', {pages}'
            citation += f' ({year}).'
        if doi:
            citation += f' doi:{doi}'
        return citation


class BibTeXManager:
    """Parse, validate, and manage a BibTeX library."""

    def __init__(self):
        self.entries: List[BibEntry] = []

    def parse_bibtex_string(self, bibtex_str: str) -> List[BibEntry]:
        """Parse a BibTeX string into BibEntry objects.

        Handles basic @type{key, field=value, ...} syntax.
        """
        entries = []
        # Match @type{citekey, ...}
        entry_pattern = re.compile(
            r'@(\w+)\{([^,]+),([^@]*)\}',
            re.DOTALL
        )
        field_pattern = re.compile(
            r'(\w+)\s*=\s*[{"](.*?)[}"]',
            re.DOTALL
        )
        for match in entry_pattern.finditer(bibtex_str):
            entry_type = match.group(1).lower()
            cite_key = match.group(2).strip()
            body = match.group(3)
            fields = {k.lower(): v.strip() for k, v in field_pattern.findall(body)}
            entries.append(BibEntry(entry_type, cite_key, fields))
        self.entries.extend(entries)
        return entries

    def validate(self) -> List[str]:
        """Return list of validation issues across all entries."""
        issues = []
        for e in self.entries:
            missing = e.missing_required()
            if missing:
                issues.append(f'[{e.cite_key}] Missing required fields: {", ".join(missing)}')
            if not e.has_doi():
                issues.append(f'[{e.cite_key}] Missing DOI')
        return issues

    def detect_duplicates(self) -> List[Tuple[str, str]]:
        """Find pairs of entries with matching titles (normalized)."""
        def normalize(title):
            return re.sub(r'[^a-z0-9]', '', title.lower())

        seen = {}
        duplicates = []
        for e in self.entries:
            title = e.fields.get('title', '')
            norm = normalize(title)
            if norm in seen:
                duplicates.append((seen[norm], e.cite_key))
            else:
                seen[norm] = e.cite_key
        return duplicates

    def summary(self) -> dict:
        type_counts = {}
        for e in self.entries:
            type_counts[e.entry_type] = type_counts.get(e.entry_type, 0) + 1
        years = [int(e.fields.get('year', 0)) for e in self.entries if e.fields.get('year', '0').isdigit()]
        return {
            'total': len(self.entries),
            'by_type': type_counts,
            'year_range': (min(years), max(years)) if years else None,
            'missing_doi': sum(1 for e in self.entries if not e.has_doi()),
            'validation_issues': len(self.validate()),
            'duplicates': len(self.detect_duplicates()),
        }


# Simulate a 6-paper BibTeX library
sample_bibtex = """
@article{needleman1970,
  author  = {Needleman, Saul B. and Wunsch, Christian D.},
  title   = {A general method applicable to the search for similarities in the amino acid sequence of two proteins},
  journal = {Journal of Molecular Biology},
  year    = {1970},
  volume  = {48},
  pages   = {443--453},
  doi     = {10.1016/0022-2836(70)90057-4}
}

@article{love2014,
  author  = {Love, Michael I. and Huber, Wolfgang and Anders, Simon},
  title   = {Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2},
  journal = {Genome Biology},
  year    = {2014},
  volume  = {15},
  pages   = {550},
  doi     = {10.1186/s13059-014-0550-8}
}

@article{jumper2021,
  author  = {Jumper, John and Evans, Richard and Pritzel, Alexander},
  title   = {Highly accurate protein structure prediction with AlphaFold},
  journal = {Nature},
  year    = {2021},
  volume  = {596},
  pages   = {583--589},
  doi     = {10.1038/s41586-021-03819-2}
}

@article{wolf2018,
  author  = {Wolf, F. Alexander and Angerer, Philipp and Theis, Fabian J.},
  title   = {SCANPY: large-scale single-cell gene expression data analysis},
  journal = {Genome Biology},
  year    = {2018},
  volume  = {19},
  pages   = {15}
}

@article{ioannidis2005,
  author  = {Ioannidis, John P. A.},
  title   = {Why Most Published Research Findings Are False},
  journal = {PLOS Medicine},
  year    = {2005},
  volume  = {2},
  pages   = {e124},
  doi     = {10.1371/journal.pmed.0020124}
}

@article{ioannidis2005dup,
  author  = {Ioannidis, John P. A.},
  title   = {Why Most Published Research Findings Are False},
  journal = {PLOS Medicine},
  year    = {2005},
  doi     = {10.1371/journal.pmed.0020124}
}
"""

mgr = BibTeXManager()
entries = mgr.parse_bibtex_string(sample_bibtex)
print(f"Parsed {len(entries)} entries")

print("\nValidation issues:")
for issue in mgr.validate():
    print(f"  {issue}")

dups = mgr.detect_duplicates()
print(f"\nDuplicates detected: {dups}")

print("\nSample APA citation:")
print(" ", entries[2].format_apa())
print("\nSample Nature citation:")
print(" ", entries[2].format_nature())

In [ ]:
s = mgr.summary()
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('BibTeX Library Analysis', fontsize=13, fontweight='bold')

# Panel 1: Library composition by entry type
ax1 = axes[0]
types = list(s['by_type'].keys())
counts = list(s['by_type'].values())
colors = plt.cm.Set2(np.linspace(0, 1, len(types)))
ax1.pie(counts, labels=types, autopct='%1.0f%%', colors=colors)
ax1.set_title('Library Composition\nby Entry Type')

# Panel 2: Year distribution
ax2 = axes[1]
years = [int(e.fields.get('year', 0)) for e in mgr.entries if e.fields.get('year', '0').isdigit()]
ax2.hist(years, bins=range(min(years), max(years)+2), color='steelblue', alpha=0.8, edgecolor='white')
ax2.set_xlabel('Year')
ax2.set_ylabel('Papers')
ax2.set_title('Publication Year Distribution')
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: Quality checklist bar chart
ax3 = axes[2]
quality_labels = ['Has DOI', 'Complete\nrequired fields', 'No duplicates']
total = len(mgr.entries)
has_doi = sum(1 for e in mgr.entries if e.has_doi())
complete = sum(1 for e in mgr.entries if not e.missing_required())
dup_keys = {key for pair in mgr.detect_duplicates() for key in pair}
no_dup = total - len(dup_keys)
quality_values = [has_doi, complete, no_dup]
bar_colors = ['#43A047' if v == total else '#E53935' for v in quality_values]
ax3.bar(quality_labels, quality_values, color=bar_colors, alpha=0.85)
ax3.axhline(total, color='gray', linestyle='--', label=f'Total ({total})')
ax3.set_ylabel('Count')
ax3.set_title('Library Quality Checklist')
ax3.legend()
ax3.set_ylim(0, total + 1)
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('bibtex_library.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Add 5 papers from Module 08 or 09 to your Zotero library (use DOI lookup). Export as BibTeX. Parse with `BibTeXManager`. Are all required fields present?

**Exercise 2:** Take any entry from `sample_bibtex` and add it twice with different cite keys. Confirm `detect_duplicates()` catches it.

**Exercise 3:** Format the BLAST paper (Altschul et al. 1990) in both APA and Nature styles, by hand and then with `BibEntry.format_apa()` / `format_nature()`. Verify they match.

**Exercise 4 (reflection):** Why is a DOI more reliable than a URL for a paper reference? What happens when a journal changes its URL structure?

## Quiz

1. What are the four required fields for a `@article` BibTeX entry?
2. How do you add a paper to Zotero by DOI?
3. What format does BibTeX use for author lists with multiple authors?
4. How do APA and Nature citation formats differ in their treatment of journal names?
5. If a paper's cite key is `love2014` but there are two papers by Love in 2014, how do you distinguish them?

## References

- Zotero documentation: https://www.zotero.org/support/
- doi2bib: https://doi2bib.org/
- BibTeX format reference: http://www.bibtex.org/Format/